# Beginner's Day - PyCon Italia 2026
*by Data Masters*  

<img src="https://s3-eu-west-1.amazonaws.com/tpd/logos/60c784605fd8980001d96a17/0x0.png" width=20 >

# Creiamo un Agente AI da zero

## Contesto del workshop

### Cosa costruiremo oggi
L'obiettivo del notebook è costruire un **Travel Agent** partendo da zero: un agente AI capace di usare strumenti Python per recuperare dati reali e combinarli in una risposta utile per chi sta organizzando un viaggio.

Alla fine del percorso l'agente saprà:

- calcolare il budget totale di un viaggio;
- cercare le coordinate di una città;
- leggere il meteo corrente tramite API;
- recuperare informazioni su un paese, come capitale, valuta e lingue;
- suggerire cosa mettere in valigia con regole Python;
- mantenere un minimo di memoria tra turni della stessa conversazione.

### Agenda
| Blocco | Tema | Perché serve |
|---|---|---|
| 1 | Ollama + LangChain | Preparare un modello locale e collegarlo al notebook |
| 2 | Primo tool Python | Trasformare una funzione in uno strumento utilizzabile dall'agente |
| 3 | Geocoding e meteo | Chiamare API REST gratuite senza API key |
| 4 | Paesi e valigia | Unire API esterne e logica deterministica |
| 5 | Travel Agent | Assemblare più tool con `create_agent()` |
| 6 | Streaming e confronti | Osservare i passaggi dell'agente in tempo reale |
| 7 | Affidabilità e memoria | Gestire errori, output strutturato e conversazioni separate |


### LLM in locale con Ollama

**Ollama è un po' come Docker per i modelli AI**. Con pochi comandi possiamo scaricare ed eseguire un modello direttamente nell'ambiente Colab, senza costruire da zero un'infrastruttura di serving.

Nel workshop questo è utile perché:

- non serve configurare una API key;
- il modello gira nel runtime del notebook;
- tutti i partecipanti possono seguire lo stesso setup;
- LangChain può parlare con Ollama via HTTP come se fosse un backend di chat.

| Ollama in locale | API cloud |
|---|---|
| Nessuna API key obbligatoria | Richiede account e chiave API |
| Dati elaborati nel runtime locale | I dati passano dal provider cloud |
| Ottimo per workshop e prototipi | Più comodo per produzione e deployment |
| Prestazioni legate alla macchina/GPU | Infrastruttura gestita dal provider |

Le prossime celle installano Ollama, avviano il server locale e scaricano il modello `gemma4:e4b`.


## Configurazione di Ollama per Colab <img src="https://ollama.com/public/ollama.png" width=20 >

In [ ]:
!sudo apt update # update dei pacchetti già installati in ambiente
!sudo apt-get install zstd # installiamo zstd, tool per la compressione e decompressione dei dati
!sudo apt install -y pciutils # installiamo pciutils per assicurarci la compatibilità con la GPU
!curl -fsSL https://ollama.com/install.sh | sh # scarichiamo ed installiamo Ollama su Colab

### Cosa succede quando avviamo `ollama serve`

Ollama espone un piccolo servizio HTTP locale. Questo servizio rimane in ascolto e riceve le richieste generate da LangChain.

```text
Notebook Colab → ollama serve → modello gemma4:e4b
```

Nel codice usiamo un thread separato perché `ollama serve` è un processo che deve restare attivo. Se lo eseguissimo in modo bloccante, il notebook resterebbe fermo su quella cella.


In [ ]:
# facciamo partire Ollama

import threading # per eseguire Ollama senza bloccare il notebook
import subprocess # per eseguire "ollama serve" come processo esterno
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [ ]:
!ollama pull gemma4:e4b
!ollama list

## 1. Langchain <img src="https://raw.githubusercontent.com/lobehub/lobe-icons/refs/heads/master/packages/static-png/dark/langchain-color.png" width=20 >

### Come LangChain si connette a Ollama

LangChain non esegue direttamente il modello: manda richieste al backend Ollama.

```text
init_chat_model("ollama:gemma4:e4b")
        ↓
LangChain crea un oggetto modello
        ↓
create_agent() userà quel modello per decidere quando chiamare i tool
```

Due funzioni saranno centrali:

- `init_chat_model(...)`: inizializza il modello di chat;
- `create_agent(...)`: costruisce un agente capace di usare il modello e una lista di tool.

Dopo l'installazione importiamo anche `@tool`, il decoratore che trasforma funzioni Python in strumenti usabili dall'agente.


In [ ]:
!pip install langchain_ollama langchain_community

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

In [ ]:
model = init_chat_model(--)

## 2. Primo tool locale: budget di viaggio

Cominciamo senza API esterne. Creiamo una funzione Python normale, poi la trasformiamo in tool.

### Cos'è un tool per un agente AI

1. **Nome**: l'agente lo usa per distinguere uno strumento dagli altri.
2. **Docstring**: spiega al modello *quando* usare quel tool e che cosa restituisce.
3. **Type hints**: aiutano il modello a passare argomenti nel formato corretto.

Un tool non è magia: è una funzione Python normale, ma impacchettata in modo che l'agente possa chiamarla durante il ragionamento.

La cella successiva implementa questo primo tool. Nota il pattern `ok=True` / `ok=False`: renderà l'agente più robusto quando incontrerà input impossibili.


In [ ]:
@tool
def calculate_trip_budget(--) -> dict:
    """Calcola il budget totale di un viaggio in euro dati giorni, budget giornaliero e costi extra."""
    if days <= 0:
        return {--}
    if daily_budget_eur < 0 or extra_costs_eur < 0:
        return {--}

    base_total = --
    total = --
    return {
        "ok": True,
        "days": days,
        "daily_budget_eur": daily_budget_eur,
        "extra_costs_eur": extra_costs_eur,
        "base_total_eur": round(base_total, 2),
        "total_eur": round(total, 2),
    }

In [ ]:
calculate_trip_budget.invoke()

### Primo agente con un solo tool

Questo agente non sa nulla di meteo o paesi. Sa solo calcolare budget.

### Il ciclo ReAct: Reason → Act → Observe

Un chatbot tradizionale risponde solo con testo. Un agente, invece, può alternare ragionamento e azioni.

```text
Reason  →  Act  →  Observe  →  Reason  →  risposta finale
```

Nel nostro caso:

- **Reason**: il modello capisce che serve un calcolo di budget;
- **Act**: chiama `calculate_trip_budget`;
- **Observe**: legge il dizionario restituito dal tool;
- **Risposta finale**: spiega il risultato all'utente.

La prossima cella crea un agente con un solo tool. È volutamente limitato: sa calcolare budget, ma non sa ancora nulla di meteo, coordinate o paesi.


In [ ]:
budget_agent = create_agent(--)

result = budget_agent.invoke(--)

print(result["messages"][-1].content)

In [ ]:
from langchain_core.runnables.graph import MermaidDrawMethod
from IPython.display import display, Image

display(
    Image(
        budget_agent.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API,
        )
    )
)

## 3. API REST in Python: coordinate città con Open-Meteo

Per sapere il meteo di una città, prima dobbiamo conoscere le coordinate.

Useremo il servizio di geocoding di Open-Meteo.

### API REST in Python

Per passare da un esempio locale a un agente utile, dobbiamo permettere al codice di leggere dati reali:

```text
requests.get(url, params=dict) → .json()
```

Nel caso del geocoding:

```text
Python requests
    ↓ GET /v1/search?name=Bologna&count=1
geocoding-api.open-meteo.com
    ↓
risultato JSON con latitudine, longitudine, paese e timezone
```

Le API meteo non lavorano con il nome della città, ma con coordinate numeriche. Prima troviamo `lat` e `lon`, poi chiediamo il meteo.


In [ ]:
import requests
from urllib.parse import quote

GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": "Bologna",
    "count": 3,
    "language": "it",
    "format": "json"
}

response = requests.get(--)
response.raise_for_status()
data = response.json()
data

### `get_json`: un wrapper riusabile

Possiamo incapsulare la chiamata HTTP in una funzione unica per evitare di riscrivere sempre le stesse tre operazioni:

1. chiamare l'endpoint con `requests.get`;
2. fermarsi se la risposta HTTP contiene un errore;
3. convertire la risposta in JSON.

Da qui in poi useremo `get_json` come base per tutti i tool che parlano con API esterne.


In [ ]:
def get_json(url: str, params: dict | None, timeout: int = 20) -> dict:
  response = requests.get(url, params=params, timeout=timeout)
  response.raise_for_status()
  return {"ok":True, "data": response.json()}

In [ ]:
get_json(--)

In [ ]:
def search_city(--) -> dict:
  ---

  return 

search_city("Bologna")


### Trasformiamo la funzione in un tool

### Da funzione Python a `search_city_tool`

La funzione `search_city(...)` fa il lavoro tecnico: chiama Open-Meteo Geocoding e pulisce il risultato.

Il tool `search_city_tool(...)`, invece, serve all'agente. La docstring gli dice chiaramente quando usarlo:

> Trova latitudine, longitudine, paese e timezone di una città. Usa questo tool prima di chiedere il meteo.

Questa frase è fondamentale: il modello non conosce il flusso delle nostre API, quindi dobbiamo spiegargli che il meteo richiede prima le coordinate.


In [ ]:
@tool
def search_city_tool(city:str) -> dict:
  """Trova latitudine, longitudine, paese e timezone di una città. Usa questo tool prima di chiedere il meteo."""
  result = search_city(city, count=1)

  if not result["ok"]:
    return result

  matches = --
  return {
        "ok": True,
        "city": matches["name"],
        "country": matches["country"],
        "country_code": matches["country_code"],
        "admin1": matches["admin1"],
        "latitude": matches["latitude"],
        "longitude": matches["longitude"],
        "timezone": matches["timezone"],
    }

search_city_tool.invoke(--)

## 4. API meteo: Open-Meteo Forecast

Ora che abbiamo le coordinate, possiamo chiedere il meteo corrente.

### Due API, un obiettivo

A questo punto colleghiamo due API gratuite e senza API key:

| Fase | Input | Output | Endpoint |
|---|---|---|---|
| Geocoding | Nome città, es. `Bologna` | Latitudine, longitudine, timezone | `geocoding-api.open-meteo.com` |
| Meteo | Latitudine + longitudine | Temperatura, pioggia, vento, codice meteo | `api.open-meteo.com` |

Il tool `get_current_weather_tool` riceve coordinate numeriche, non il nome della città. L'agente dovrà quindi imparare questa sequenza:

```text
nome città → search_city_tool → coordinate → get_current_weather_tool → meteo
```

La separazione in due tool rende esplicito il workflow e permette di riusare i singoli pezzi anche in altri agenti.


In [ ]:
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"

WEATHER_CODE_LABELS = {
    0: "cielo sereno",
    1: "prevalentemente sereno",
    2: "parzialmente nuvoloso",
    3: "coperto",
    45: "nebbia",
    48: "nebbia con brina",
    51: "pioviggine leggera",
    53: "pioviggine moderata",
    55: "pioviggine intensa",
    61: "pioggia leggera",
    63: "pioggia moderata",
    65: "pioggia intensa",
    71: "neve leggera",
    73: "neve moderata",
    75: "neve intensa",
    80: "rovesci leggeri",
    81: "rovesci moderati",
    82: "rovesci violenti",
    95: "temporale",
    96: "temporale con grandine leggera",
    99: "temporale con grandine forte",
}


def weather_label(code: int | None) -> str:
    if code is None:
        return "condizione non disponibile"
    return WEATHER_CODE_LABELS.get(--)

In [ ]:
def get_current_weather(--) -> dict:
    """Ottiene il meteo corrente da Open-Meteo date latitudine e longitudine."""
    params = ---
    result = get_json(--)

    if not result["ok"]:
        return result

    data = --
    current = --
    units = --

    code = current.get("weather_code")
    return {
        "ok": True,
        "timezone": data.get("timezone"),
        "time": current.get("time"),
        "temperature": current.get("temperature_2m"),
        "temperature_unit": units.get("temperature_2m", "°C"),
        "humidity_percent": current.get("relative_humidity_2m"),
        "precipitation_mm": current.get("precipitation"),
        "rain_mm": current.get("rain"),
        "wind_speed": current.get("wind_speed_10m"),
        "wind_speed_unit": units.get("wind_speed_10m", "km/h"),
        "weather_code": code,
        "condition": weather_label(code),
    }

# Test manuale: Roma circa 41.9, 12.5
get_current_weather(44.4, 11.3)

In [ ]:
@tool
def get_current_weather_tool(latitude: float, longitude: float) -> dict:
  """Ottiene il meteo corrente per una posizione geografica usando latitudine e longitudine."""
  return get_current_weather(latitude, longitude)

get_current_weather_tool.invoke(--)

## 5. API paesi: REST Countries

### Paesi, capitali, lingue e valute

Per una scheda viaggio il meteo non basta. Introduciamo un tool basato su REST Countries, utile per recuperare informazioni pratiche sul paese di destinazione.

Il tool `get_country_info_tool(country_name: str)` deve restituire dati come:

- capitale;
- valuta;
- lingue;
- popolazione;
- timezone;
- regione e sotto-regione.

Un dettaglio didattico importante: il tool accetta nomi in italiano o inglese. Nel codice gestiamo quindi anche alcune traduzioni comuni, per rendere più naturale l'interazione con l'utente.


In [ ]:
REST_COUNTRIES_BASE_URL = "https://restcountries.com/v3.1"


def get_country_info(country_name: str) -> dict:
    """Recupera informazioni di base su un paese usando REST Countries."""
    fields = "name,capital,currencies,languages,population,region,subregion,timezones,cca2,flags"

    # Prima proviamo l'endpoint translation, utile per nomi in italiano come "Spagna".
    # Poi proviamo l'endpoint name, utile per nomi in inglese come "Spain".
    endpoints = [---]

    last_error = None
    for url in endpoints:
        result = ----

            languages = country.get("languages", {}) or {}

            return {
                "ok": True,
                "input": country_name,
                "common_name": country.get("name", {}).get("common"),
                "official_name": country.get("name", {}).get("official"),
                "country_code": country.get("cca2"),
                "capital": country.get("capital", []),
                "currencies": currency_list,
                "languages": list(languages.values()),
                "population": country.get("population"),
                "region": country.get("region"),
                "subregion": country.get("subregion"),
                "timezones": country.get("timezones", []),
                "flag_png": country.get("flags", {}).get("png"),
            }
        last_error = result.get("error")

    return {
        "ok": False,
        "error": f"Paese non trovato o API non disponibile per: {country_name}. Ultimo errore: {last_error}",
    }

get_country_info("Spagna")

In [ ]:
@tool
def get_country_info_tool(country_name: str) -> dict:
    """Recupera capitale, valuta, lingue, popolazione e timezone di un paese. Accetta nomi in italiano o inglese."""
    return get_country_info(country_name)

get_country_info_tool.invoke(--)

## 6. Tool per consigli deterministici: cosa mettere in valigia

Non tutto deve essere deciso dal modello. A volte conviene codificare una logica semplice in Python.

Creiamo un tool che con dati di temperatura, pioggia e vento, suggerisce cosa portare.

### Quando il codice batte il modello

> Se la logica è fatta di regole fisse, codificala in Python. Non lasciare decidere tutto al modello.

Esempio:

```text
temperatura < 8°C  → cappotto, sciarpa, guanti
pioggia > 0 mm     → ombrello, scarpe impermeabili
vento > 30 km/h    → giacca antivento
```

Il modello è bravo a capire la richiesta e a orchestrare i tool. Le decisioni deterministiche, però, sono spesso più sicure se scritte con `if/else`.

La prossima cella implementa questa idea con `suggest_packing_items`.


In [ ]:
@tool
def suggest_packing_items(--) -> dict:
    """Suggerisce oggetti da mettere in valigia in base a temperatura, precipitazioni e vento."""
    items = []

    ---

    return {
        "ok": True,
        "temperature_c": temperature_c,
        "precipitation_mm": precipitation_mm,
        "wind_speed_kmh": wind_speed_kmh,
        "recommended_items": sorted(set(items)),
    }

suggest_packing_items.invoke({"temperature_c": 18, "precipitation_mm": 2, "wind_speed_kmh": 12})

## 7. Costruiamo il Travel Agent con `create_agent()`

Ora combiniamo tutti i tool.

Il nostro agente può:

- trovare coordinate di una città
- ottenere il meteo corrente
- recuperare informazioni sul paese
- calcolare budget
- suggerire cosa mettere in valigia

### Architettura del Travel Agent

Ora assembliamo i pezzi.

Tool disponibili:

- `search_city_tool`: cerca coordinate e paese a partire da una città;
- `get_current_weather_tool`: legge il meteo corrente dalle coordinate;
- `get_country_info_tool`: recupera capitale, valuta, lingue e altre informazioni;
- `calculate_trip_budget`: calcola il budget totale;
- `suggest_packing_items`: suggerisce cosa portare in valigia.

Con `create_agent()` diamo al modello:

1. un modello di chat;
2. una lista di tool;
3. un system prompt con le regole del gioco.

Il system prompt è il contratto operativo dell'agente: gli dice di rispondere in italiano, usare i tool per dati reali, non inventare e gestire gli errori `ok=False`.


In [ ]:
TRAVEL_AGENT_PROMPT = ---

travel_tools = [---]

travel_agent = create_agent(---)

### Scheda completa del viaggio

Mettiamo insieme tutti i pezzi: l'utente chiede una mini scheda viaggio e l'agente deve decidere quali dati servono.

Per rispondere bene dovrebbe:

1. riconoscere la città di destinazione;
2. usare il geocoding per ottenere le coordinate;
3. usare il meteo corrente;
4. recuperare lingua e valuta del paese;
5. calcolare il budget totale;
6. suggerire oggetti da mettere in valigia;
7. organizzare tutto in una risposta leggibile.

La cosa interessante non è solo la risposta finale, ma la sequenza di tool call che porta alla risposta. Per questo subito dopo la demo stamperemo anche tutti i messaggi intermedi prodotti dall'agente.


In [ ]:
question = """
Vado 3 giorni a Barcellona con 80 euro al giorno e 60 euro di costi extra.
Mi fai una mini scheda con meteo attuale, lingua, valuta, budget totale e cosa mettere in valigia?
"""

result = travel_agent.invoke(---)

result

In [ ]:
# Mostra tutti i messaggi prodotti: utile per vedere tool call e risultati.

for i, message in enumerate(result["messages"], start=1):
    print("=" * 80)
    print(f"Messaggio {i}: {type(message).__name__}")
    if getattr(message, "tool_calls", None):
        print("Tool calls:")
        for call in message.tool_calls:
            print(call)
    content = getattr(message, "content", "")
    print(content)

### `invoke()` vs `stream()`

Due modi di usare un agente:

| `invoke()` | `stream()` |
|---|---|
| Aspetta il risultato finale | Mostra gli step mentre accadono |
| Ritorna un dizionario con i messaggi | Produce eventi progressivi |
| Più semplice per pipeline batch | Più utile per debug e interfacce utente |
| Nasconde il percorso intermedio | Permette di vedere tool call e osservazioni |

Con `stream()` possiamo osservare l'agente mentre lavora. Nella prossima demo gli chiediamo di confrontare Roma e Berlino: l'agente dovrà cercare entrambe le città, recuperare due meteo e poi confrontare le temperature.


In [ ]:
stream_question = "Fa più freddo ora a Roma o a Berlino? Rispondi confrontando le temperature."

for step in travel_agent.stream(---):
    message = step["messages"][-1]
    message.pretty_print()

## 8. Affidabilità: input ambigui ed errori

Un agente reale deve cavarsela anche quando la domanda è incompleta, ambigua o contiene dati impossibili.

### Affidabilità: agenti che non si rompono

Un agente reale deve gestire anche input strani, incompleti o impossibili:

- `Vado ad Atlantide, che tempo fa?`
- `Spendo per -2 giorni a 100€/giorno?`
- `Vado a Parigi, serve l'ombrello?`

Il pattern che usiamo nei tool è:

```python
{"ok": False, "error": "..."}
```

In questo modo il tool non manda in crash il notebook. Restituisce invece un errore leggibile, che l'agente può spiegare all'utente.

Questo è uno dei passaggi più importanti del workshop: un agente utile non deve solo funzionare nei casi felici, ma deve fallire bene quando i dati non sono validi.


In [ ]:
questions = [---]

for q in questions:
    print("=" * 100)
    print("DOMANDA:", q)
    result = travel_agent.invoke(---)
    print(result["messages"][-1].content)

## 9. Miglioriamo l'output: formato scheda viaggio

Possiamo chiedere all'agente di rispondere sempre con una struttura precisa.

Questo non garantisce al 100% un formato perfetto, ma aiuta molto.

### Output strutturato

Aggiungiamo al system prompt un template fisso. Questo non garantisce al 100% che il modello rispetti sempre il formato, ma aumenta molto la probabilità di ottenere risposte coerenti.

Esempio di struttura:

```markdown
## Scheda viaggio
- Destinazione:
- Meteo attuale:
- Lingua/e:
- Valuta:
- Budget:
- Cosa mettere in valigia:
- Nota utile:
```

Questo approccio è utile quando vogliamo mostrare la risposta in una UI, salvarla in un report o confrontare più risposte tra loro.


In [ ]:
STRUCTURED_TRAVEL_PROMPT = TRAVEL_AGENT_PROMPT + """

Quando prepari una scheda viaggio, usa questo formato:

## Scheda viaggio
- Destinazione:
- Meteo attuale:
- Lingua/e:
- Valuta:
- Budget:
- Cosa mettere in valigia:
- Nota utile:
"""

structured_travel_agent = create_agent(---)

result = structured_travel_agent.invoke({---})

print(result["messages"][-1].content)

## 10. Diamo una memoria al nostro agente

Finora ogni chiamata all'agente era indipendente: l'input conteneva tutto quello che serviva.

In una vera chat, invece, vogliamo che l'agente ricordi quello che l'utente ha detto nei turni precedenti della stessa conversazione. In LangChain questa memoria di breve periodo si ottiene aggiungendo un **checkpointer** all'agente.

Idea chiave:

- `checkpointer`: salva lo stato dell'agente dopo ogni turno e dopo ogni tool call
- `thread_id`: identifica una conversazione specifica
- stesso `thread_id` = l'agente continua la stessa conversazione
- `thread_id` diverso = conversazione separata, senza memoria condivisa

> In Colab useremo `InMemorySaver`: è perfetto per il workshop, ma la memoria vive solo nel runtime corrente. Se riavvii il notebook, viene persa. In produzione useresti un checkpointer persistente, per esempio database-backed.

### Memoria con `InMemorySaver` e `thread_id`

Finora ogni chiamata era indipendente. Con la memoria vogliamo che l'agente possa continuare una conversazione.

Introduciamo due concetti:

- `checkpointer`: salva lo stato dell'agente;
- `thread_id`: identifica una conversazione.

```text
stesso thread_id  → stessa conversazione, memoria condivisa
thread_id diverso → conversazione nuova, memoria separata
```

Per il workshop useremo `InMemorySaver`, che salva tutto in RAM. È perfetto in Colab, ma ha due limiti:

1. se il runtime si riavvia, la memoria viene persa;
2. il modello non può vedere infinite informazioni, perché esiste sempre una context window con un limite di token.

In produzione useremmo un checkpointer persistente, ad esempio basato su database.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

structured_travel_agent = create_agent(---)

config_a = {"configurable": {"thread_id": "1"}}

result = structured_travel_agent.invoke(---)

print(result["messages"][-1].content)

In [ ]:
result = structured_travel_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "qual è l'ultima scheda che hai preparato?"
    }]
  },
  ---)

print(result["messages"][-1].content)

In [ ]:
# proviamo con un nuovo thread
config_b = {"configurable": {"thread_id": "2"}}

result = structured_travel_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "qual è l'ultima scheda che hai preparato?"
    }]
  },
  ---)

print(result["messages"][-1].content)

In [ ]:
# vediamo cosa c'è dentro la memoria dell'agente
snapshot = structured_travel_agent.get_state(config_a)
saved_messages = snapshot.values.get("messages", [])

print("Messaggi salvati nel thread:", len(saved_messages))
print("\nUltimi messaggi salvati:")
for message in saved_messages[-6:]:
    print("-", type(message).__name__, "|", str(getattr(message, "content", ""))[:180].replace("\n", " "))


## Conclusioni

In questo workshop hai visto cinque pattern fondamentali per costruire agenti AI con Python:

1. **`@tool`**: una funzione diventa uno strumento quando ha nome, docstring e type hints chiari.
2. **ReAct**: l'agente alterna ragionamento, azione e osservazione.
3. **API REST**: con `requests.get()` puoi collegare l'agente a dati reali.
4. **`create_agent()`**: modello, tool e system prompt vengono assemblati in un unico agente.
5. **`thread_id`**: conversazioni diverse possono avere memorie separate.

### Prossimi passi

Per continuare puoi:

- aggiungere un nuovo tool, per esempio conversione valuta, ricerca eventi o stima emissioni CO₂;
- cambiare dominio, passando da viaggi a logistica, HR, customer care o education;
- sostituire `InMemorySaver` con una memoria persistente;
- definire output più rigorosi se devi integrare l'agente in una applicazione reale.

> Messaggio chiave: un agente AI diventa utile quando combina il linguaggio naturale del modello con strumenti Python semplici, testabili e affidabili.


# Codice Sconto del 50%

<img src="qr_code.png" width=400 >

